# Demo suy luận THẬT — CountGD++ trên ảnh mới (CS331), Kaggle T4

Notebook này chạy model trên GPU của Kaggle rồi mở giao diện web qua link **https://….gradio.live** để mở trên trình duyệt.
Bạn dán ảnh bằng Ctrl+V, nhập tên vật, bấm **Chạy**. Model chạy thật trên ảnh đó cho cả bốn cách: B0, B2, P+F, C6.

**Cài đặt phiên (bên phải màn hình, mục Session options):**
- **Accelerator: GPU T4 x2**.
- **Internet: On**. Kaggle yêu cầu tài khoản đã xác minh số điện thoại.
- (Nếu gdown báo quá lượt tải) **Add Input** → dataset riêng chứa `countgd_plusplus.pth`. Đây chính là dataset đã gắn khi chạy test 11/11.

**Chạy:** dùng **Run All** trong chế độ chỉnh sửa (phiên tương tác), **không** dùng "Save & Run All". Cách sau chạy nền, không mở được link.
- Lần đầu khoảng 10–15 phút: cài môi trường và build CUDA ops.
- Ô cuối in **link** và **mật khẩu** (user `cs331`). Giữ ô đó chạy và giữ tab Kaggle mở trong lúc demo.

**Giới hạn:**
- Link gradio.live sống tối đa 72 giờ nhưng chết ngay khi phiên Kaggle tắt.
- Phiên tương tác tắt sau khoảng 12 giờ, hoặc sớm hơn nếu bỏ tab lâu không thao tác.
- GPU miễn phí khoảng 30 giờ/tuần.
- Mỗi lần mở phiên mới thì link mới.

**Không đổi điều kiện so với lần test 11/11:**
- Cùng checkpoint (sha256 kiểm theo audit 14/10), cùng `process_folder.py` đã pin.
- Cùng môi trường: ô 3 chép nguyên văn từ notebook W8.
- Seed 42; τ_sample 0.23; P(τ 0.20, M 5); F với bounds/threshold **riêng** của P+F và C6 từ `freeze_v2_20261111.json`.
- Ô 6 chạy lại 3 query test đã có kết quả để xác nhận wrapper khớp số đếm chính thức. Đây chỉ là kiểm wrapper, **không** phải chạy lại test.

Ảnh dùng trong demo **không** đưa vào bảng kết quả nghiên cứu. Log từng lần chạy: `/kaggle/working/demo_logs/requests.jsonl`.

In [ ]:
# Ô 1 — thiết lập chung (cùng tên biến với notebook W8 để ô 3 chạy nguyên văn)
import datetime, glob, hashlib, json, os, pathlib, re, secrets, shlex, shutil, subprocess, sys, time
REPO_URL = "https://github.com/minhtridaoo/zero-shot-counting.git"
REPO_REF = "974048f"
FREEZE_SHA = "64e89ef53eab306fdbb33cfc39797749ff17cc8635f46ae381d96e45c4f6736d"
ROOT = pathlib.Path("/tmp/CS331")
PY = "/tmp/venv310/bin/python"
OUT = pathlib.Path("/kaggle/working/demo_out"); OUT.mkdir(parents=True, exist_ok=True)
LOGS = pathlib.Path("/kaggle/working/demo_logs"); LOGS.mkdir(parents=True, exist_ok=True)
DEMO_PASSWORD = os.environ.get("DEMO_PASSWORD") or secrets.token_urlsafe(6)   # đổi ở đây nếu muốn mật khẩu cố định
ENV = os.environ.copy()
ENV.update(CUDA_VISIBLE_DEVICES="0", MPLBACKEND="Agg", PYTHONUNBUFFERED="1", GRADIO_ANALYTICS_ENABLED="False")
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"Lệnh lỗi (mã {p.returncode}), xem các dòng ngay phía trên. Dừng.")
def file_hash(path, algo="sha256"):
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()
gpu = subprocess.check_output("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", shell=True, text=True)
print(gpu)
assert "T4" in gpu.splitlines()[0], "Cần GPU T4 (Session options > Accelerator > GPU T4 x2). Dừng."

In [ ]:
# Ô 2 — lấy code ở commit cố định; kiểm freeze v2
if not ROOT.exists():
    sh(f"git clone -q {REPO_URL} {ROOT}")
sh(f"git fetch -q origin && git checkout -q {REPO_REF}", cwd=ROOT)
COMMIT = subprocess.check_output("git rev-parse HEAD", shell=True, cwd=ROOT, text=True).strip()
print("commit", COMMIT)
assert file_hash(ROOT / "results/countgdpp/freeze_v2_20261111.json") == FREEZE_SHA, "freeze v2 khác sha256. Dừng."
print("freeze v2 khớp sha256")

In [ ]:
# Ô 3 — môi trường /tmp/venv310 giống Colab đã qua Kiểm A/C (torch 2.5.1+cu121, transformers 4.44.2, numpy 2.2.6, pillow 10.4.0)
# và build CUDA ops (MultiScaleDeformableAttention) cho T4 (sm 7.5). Dùng nvcc 12.x có sẵn của Kaggle nếu có, không thì tải CUDA 12.1.
sh("pip install -q uv")
if not os.path.exists(PY):
    sh("uv venv /tmp/venv310 --python 3.10")
sh(f"uv pip install -q -p {PY} torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121")
sh(f"uv pip install -q -p {PY} transformers==4.44.2 timm==1.0.30 numpy==2.2.6 pillow==10.4.0 scipy termcolor addict yapf==0.40.1"
   " opencv-python-headless pycocotools matplotlib setuptools wheel ninja")
nvccs = sorted(set(filter(None, [shutil.which("nvcc"), *glob.glob("/usr/local/cuda*/bin/nvcc")])))
cuda_home = None
for nv in nvccs:
    if re.search(r"release 12\.", subprocess.run([nv, "--version"], capture_output=True, text=True).stdout):
        cuda_home = str(pathlib.Path(nv).resolve().parent.parent); break
if cuda_home is None:
    cuda_home = "/tmp/cuda-12.1"
    if not os.path.exists(f"{cuda_home}/bin/nvcc"):
        sh("wget -q -O /tmp/cuda121.run https://developer.download.nvidia.com/compute/cuda/12.1.1/local_installers/cuda_12.1.1_530.30.02_linux.run")
        sh(f"sh /tmp/cuda121.run --silent --toolkit --toolkitpath={cuda_home} --override --no-man-page && rm -f /tmp/cuda121.run")
sh(f"{cuda_home}/bin/nvcc --version | tail -1")
sh("apt-get -qq update && apt-get -qq install -y gcc-11 g++-11 > /dev/null")
ENV.update(CUDA_HOME=cuda_home, PATH=f"{cuda_home}/bin:/tmp/venv310/bin:" + ENV["PATH"], CC="gcc-11", CXX="g++-11", TORCH_CUDA_ARCH_LIST="7.5")
sh("rm -rf /tmp/ops_build /tmp/wheels && cp -r " + str(ROOT / "CountGDPlusPlus/models/GroundingDINO/ops") + " /tmp/ops_build")
sh("rm -rf /tmp/ops_build/build /tmp/ops_build/dist /tmp/ops_build/*.egg-info")
sh(f"cd /tmp/ops_build && {PY} setup.py -q bdist_wheel -d /tmp/wheels")
wheel = sorted(p for p in glob.glob("/tmp/wheels/*.whl") if os.path.basename(p).lower().startswith("multiscaledeformableattention-"))[-1]
sh(f"uv pip install -q -p {PY} --reinstall {wheel}")
probe = ("import sys, torch, transformers, numpy, PIL, MultiScaleDeformableAttention; "
         "assert torch.__version__.startswith('2.5.1'), torch.__version__; "
         "print('python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__, "
         "'| numpy', numpy.__version__, '| pillow', PIL.__version__, '| GPU', torch.cuda.get_device_name())")
sh(f"{PY} -c {shlex.quote(probe)}")
sh(f"uv pip freeze -p {PY} > {OUT}/packages.txt")

# Thêm cho demo: Gradio, giữ nguyên các bản đã pin của môi trường test (không nâng cấp torch/transformers/numpy/pillow)
sh(f'uv pip install -q -p {PY} "gradio==5.50.0" "huggingface_hub<1.0" transformers==4.44.2 numpy==2.2.6 pillow==10.4.0')
sh(f"{PY} -c " + shlex.quote("import torch, transformers, numpy, PIL, gradio; print('sau khi cài gradio: torch', torch.__version__, '| transformers', transformers.__version__, '| numpy', numpy.__version__, '| pillow', PIL.__version__, '| gradio', gradio.__version__)"))

In [ ]:
# Ô 4 — checkpoint CountGD++ (kiểm sha256 đã audit 14/10), BERT, và ảnh MUCCA (chỉ để ô 6 kiểm wrapper)
CKPT = ROOT / "CountGDPlusPlus/checkpoints/countgd_plusplus.pth"
EXPECTED = json.loads((ROOT / "results/countgdpp/checkpoint_audit_2026-10-14.json").read_text())["checkpoint_sha256"]
if not CKPT.exists():
    local = glob.glob("/kaggle/input/**/countgd_plusplus.pth", recursive=True)
    if local:
        shutil.copy(local[0], CKPT)
    else:
        sh(f"uv pip install -q -p {PY} gdown")
        sh(f'/tmp/venv310/bin/gdown "https://drive.google.com/uc?id=1j6N22TtKu2NVcKpgfrf-sJHGeLDqs9hs" -O {CKPT}')
assert file_hash(CKPT) == EXPECTED, "checkpoint sha256 sai"
print("Checkpoint đúng bản đã audit 14/10.")
BERT = ROOT / "CountGDPlusPlus/checkpoints/bert-base-uncased"
if not any((BERT / n).exists() for n in ("model.safetensors", "pytorch_model.bin")):
    sh(f"{PY} download_bert.py", cwd=ROOT / "CountGDPlusPlus")
print("BERT:", sorted(p.name for p in BERT.iterdir()))
MUCCA = ROOT / "data_external/mucca_zenodo_19231375/MUCCA_1.0"
if not (MUCCA / "images/mcac_12.jpg").exists():
    sh('wget -q -O /tmp/MUCCA_1.0.zip "https://zenodo.org/api/records/19231375/files/MUCCA_1.0.zip/content"')
    assert file_hash("/tmp/MUCCA_1.0.zip", "md5") == "b182e445445dad017493ffb35803f816", "md5 MUCCA sai"
    MUCCA.mkdir(parents=True, exist_ok=True)
    sh(f"unzip -q -o /tmp/MUCCA_1.0.zip -d {MUCCA}")
print("Ảnh MUCCA:", len(list((MUCCA / "images").glob("*.jpg"))))

In [ ]:
# Ô 5 — ghi code demo (nhúng từ máy Trí, kiểm sha256)
DEMO_FILES = {"live_engine.py": "\"\"\"Suy luận THẬT trên ảnh mới cho demo: B0, B2, P+F, C6 với cấu hình đã khóa (freeze v2).\n\nKhông sửa file gốc. Ghép lại đúng các bước của pipeline đã chạy test 11/11:\n  - nạp model như experiments/countgdpp/run_b2_verified_pseudo.py: lấy định nghĩa hàm từ CountGDPlusPlus/process_folder.py\n    (bản pin, kiểm hash), checkpoint đã audit 14/10, seed 42 trước lượt 1, khôi phục RNG về cuối lượt 1 trước mỗi lượt 2;\n  - B0 = lượt 1 (giữ box có điểm > 0.23);\n  - B2 = lượt 2 với top-3 pseudo-exemplar gốc (run_b2_pilot.select_pseudo); lượt 1 không có box thì B2 = B0;\n  - P(τ 0.20, M 5) = CLIP kiểm M ứng viên (pseudo_clip_verifier.top_candidates/choose/ClipVerifier), lượt 2 với các mẫu\n    còn lại; không còn mẫu thì = B0; nếu bộ mẫu trùng B2 thì dùng lại đúng lượt 2 đó (runner cũng cache như vậy);\n  - F = 0.5·norm(điểm CountGD++) + 0.5·norm(cosine CLIP của box cuối), bounds/threshold lấy từ freeze cho TỪNG cấu hình:\n    P+F = nguồn P(τ 0.20) + F(main), C6 = nguồn B0 + F(C6). CLIP của box cuối dùng cùng crop/batch/mẫu câu với\n    p4_clip_score.py (ClipVerifier.score, đã qua Kiểm A trên Kaggle).\nLượt 1 chạy MỘT lần cho mỗi query và dùng chung cho cả bốn cách. Model nạp một lần khi khởi động.\n\"\"\"\nfrom __future__ import annotations\n\nimport ast\nimport hashlib\nimport json\nimport os\nimport platform\nimport random\nimport sys\nimport time\nimport uuid\nfrom datetime import datetime, timedelta, timezone\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[2]\nsys.path.insert(0, str(ROOT / \"experiments/countgdpp\"))\n\nfrom run_b0_batch import REPO, SOURCE_HASH, sha  # noqa: E402\nfrom run_b2_pilot import select_pseudo  # noqa: E402\nfrom run_b2_verified_pseudo import source_sha, MAX_PSEUDO, TAU  # noqa: E402\nfrom pseudo_clip_verifier import MODEL_ID, REVISION, TEMPLATE, ClipVerifier, choose, top_candidates  # noqa: E402\n\nFREEZE = ROOT / \"results/countgdpp/freeze_v2_20261111.json\"\nFREEZE_SHA = \"64e89ef53eab306fdbb33cfc39797749ff17cc8635f46ae381d96e45c4f6736d\"\nAUDIT = ROOT / \"results/countgdpp/checkpoint_audit_2026-10-14.json\"\nMETHODS = (\"B0\", \"B2\", \"P+F\", \"C6\")\nTZ = timezone(timedelta(hours=7))\n\n\ndef norm(v, b):  # giống apply_freeze_v2.norm\n    lo, hi = b\n    return .5 if lo == hi else max(0., min(1., (v - lo) / (hi - lo)))\n\n\ndef iou(a, b):  # giống evaluation/test_xy_overlap_analysis.iou\n    ix = max(0., min(a[2], b[2]) - max(a[0], b[0])); iy = max(0., min(a[3], b[3]) - max(a[1], b[1]))\n    inter = ix * iy\n    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter\n    return inter / ua if ua > 0 else 0.\n\n\ndef shared_flags(boxes, others, t=0.5):\n    return [any(iou(b, o) >= t for o in others) for b in boxes]\n\n\ndef parse_queries(text: str, max_q: int = 3):\n    raw = [q.strip() for part in (text or \"\").replace(\";\", \",\").splitlines() for q in part.split(\",\")]\n    qs = [q for q in raw if q]\n    if not qs:\n        raise ValueError(\"Chưa nhập tên vật cần đếm. Ví dụ: bowls, plates\")\n    bad = [q for q in qs if any(ch in q for ch in \".?!\")]\n    if bad:\n        raise ValueError(f\"Tên vật không được chứa dấu chấm/chấm hỏi/chấm than (CountGD++ dùng dấu chấm để tách câu): {bad}\")\n    if len(set(q.lower() for q in qs)) != len(qs):\n        raise ValueError(\"Có tên vật bị lặp.\")\n    if len(qs) > max_q:\n        raise ValueError(f\"Tối đa {max_q} tên vật mỗi lần chạy (đang nhập {len(qs)}).\")\n    return qs\n\n\ndef parse_gt(text: str, n: int):\n    \"\"\"GT tùy chọn, cùng thứ tự với tên vật. Ô trống hoặc '-' = không có GT.\"\"\"\n    if not (text or \"\").strip():\n        return [None] * n\n    parts = [p.strip() for p in text.replace(\";\", \",\").split(\",\")]\n    if len(parts) != n:\n        raise ValueError(f\"GT phải có đúng {n} giá trị theo thứ tự tên vật (dùng '-' nếu không biết), đang có {len(parts)}.\")\n    out = []\n    for p in parts:\n        if p in (\"\", \"-\"):\n            out.append(None)\n        elif p.isdigit():\n            out.append(int(p))\n        else:\n            raise ValueError(f\"GT phải là số nguyên không âm hoặc '-': '{p}'\")\n    return out\n\n\nclass LiveEngine:\n    def __init__(self, device: str = \"cuda\", log_dir: Path | None = None):\n        import numpy as np\n        import torch\n        self.np, self.torch = np, torch\n        t0 = time.perf_counter()\n        # ---- kiểm checkpoint, code gốc, freeze (giống runner)\n        audit = json.loads(AUDIT.read_text())\n        ckpt = REPO / \"checkpoints/countgd_plusplus.pth\"\n        if not ckpt.exists():\n            raise FileNotFoundError(f\"Thiếu checkpoint {ckpt}\")\n        self.ckpt_sha = sha(ckpt)\n        if self.ckpt_sha != audit[\"checkpoint_sha256\"] or audit.get(\"review_decision\") != \"compatible_with_pinned_B0_app\":\n            raise ValueError(\"Checkpoint khác bản đã audit 14/10\")\n        if source_sha(REPO / \"process_folder.py\") != SOURCE_HASH:\n            raise ValueError(\"process_folder.py khác bản pin\")\n        for src, digest in audit[\"review_source_sha256\"].items():\n            if source_sha(REPO / src) != digest:\n                raise ValueError(f\"{src} khác bản đã audit\")\n        if sha(FREEZE) != FREEZE_SHA:\n            raise ValueError(\"freeze_v2 khác sha256 đã khóa\")\n        fz = json.loads(FREEZE.read_text(encoding=\"utf-8\"))\n        assert fz[\"status\"] == \"FROZEN_V2\"\n        a = fz[\"runner\"][\"args\"]\n        assert a[\"tau_pseudo\"] == [0.2] and a[\"m\"] == 5 and fz[\"runner\"][\"tau_sample\"] == TAU == .23 and fz[\"runner\"][\"max_pseudo\"] == MAX_PSEUDO == 3\n        assert fz[\"runner\"][\"clip_model\"] == MODEL_ID and fz[\"runner\"][\"clip_revision\"] == REVISION and fz[\"runner\"][\"clip_template\"] == TEMPLATE\n        self.tau_pseudo, self.m = a[\"tau_pseudo\"][0], a[\"m\"]\n        self.f_main = fz[\"main\"]; self.f_c6 = fz[\"companions_report_only\"][\"C6 = B0+F\"]\n        assert self.f_main[\"source\"] == \"tau0.200\" and self.f_main[\"filter\"] == \"F\"\n        assert self.f_c6[\"source\"] == \"B0\" and self.f_c6[\"filter\"] == \"F\"\n        # ---- nạp model một lần (giống run_b2_verified_pseudo.main)\n        if device == \"cuda\" and not torch.cuda.is_available():\n            raise RuntimeError(\"Không có CUDA. Demo chính cần GPU (Kaggle T4); --device cpu chỉ để kiểm chức năng.\")\n        self.cwd0 = os.getcwd()\n        sys.path.insert(0, str(REPO)); os.chdir(REPO)\n        tree = ast.parse((REPO / \"process_folder.py\").read_text())\n        tree.body = [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom, ast.FunctionDef))]\n        ns = {\"__name__\": \"pinned_stock_definitions\"}\n        exec(compile(tree, str(REPO / \"process_folder.py\"), \"exec\"), ns)\n        self.ns = ns\n        self.device = torch.device(device)\n        model, self.transform = ns[\"build_model_and_transforms\"](ns[\"build_arg_parser\"]().parse_args([]))\n        self.model = model.to(self.device)\n        self.clip = ClipVerifier(self.device)\n        original = ns[\"get_boxes_from_prediction\"]; self._cap = {}\n\n        def observe(output, text, keywords=\"\", conf_thresh=TAU):\n            returned = original(output, text, keywords, conf_thresh)\n            self._cap.update(scores=output[\"pred_logits\"][0].sigmoid().detach().cpu(),\n                             boxes=output[\"pred_boxes\"][0].detach().cpu(),\n                             ids=output[\"input_ids\"][0].detach().cpu().tolist())\n            return returned\n        ns[\"get_boxes_from_prediction\"] = observe\n        self.load_s = round(time.perf_counter() - t0, 1)\n        self.env = dict(device=torch.cuda.get_device_name() if self.device.type == \"cuda\" else f\"CPU ({platform.processor() or platform.machine()})\",\n                        torch=torch.__version__, transformers=self.clip.transformers_version, python=sys.version.split()[0],\n                        checkpoint_sha256=self.ckpt_sha, freeze_sha256=FREEZE_SHA, clip_model=MODEL_ID, clip_revision=self.clip.revision,\n                        process_folder_sha256=SOURCE_HASH)\n        self.config = dict(tau_sample=TAU, tau_final=TAU, max_pseudo=MAX_PSEUDO, tau_pseudo=self.tau_pseudo, m=self.m,\n                           seed=42, resize=dict(short_side=800, max_size=1333), clip_template=TEMPLATE,\n                           F_PplusF=dict(bounds=self.f_main[\"bounds\"], threshold=self.f_main[\"threshold\"]),\n                           F_C6=dict(bounds=self.f_c6[\"bounds\"], threshold=self.f_c6[\"threshold\"]))\n        self.log_dir = Path(log_dir) if log_dir else ROOT / \"demo/live/logs\"\n        self.log_dir.mkdir(parents=True, exist_ok=True)\n        self.log_path = self.log_dir / \"requests.jsonl\"\n\n    # ------------------------------------------------------------ helpers\n    def _sync(self):\n        if self.device.type == \"cuda\":\n            self.torch.cuda.synchronize()\n\n    def _predict(self, image, prompt, exemplars):\n        np = self.np; ns = self.ns\n        prompts = {\"image\": image, \"points\": [[b[0], b[1], 2.0, b[2], b[3], 3.0] for b in exemplars]}\n        b, s = ns[\"predict\"](self.model, self.transform, image, prompt, prompts, [], [], self.device, TAU)\n        x, y, w, h = ns[\"get_box_coords_from_boxes\"](image, b)\n        return np.stack((x, y, x + w, y + h), axis=1).reshape(-1, 4), np.asarray(s).reshape(-1), dict(self._cap)\n\n    def _rng_state(self):\n        t = self.torch\n        return (t.get_rng_state(), t.cuda.get_rng_state_all() if self.device.type == \"cuda\" else None,\n                self.np.random.get_state(), self.ns[\"random\"].getstate(), random.getstate())\n\n    def _set_rng(self, st):\n        t = self.torch\n        t.set_rng_state(st[0])\n        if st[1] is not None:\n            t.cuda.set_rng_state_all(st[1])\n        self.np.random.set_state(st[2]); self.ns[\"random\"].setstate(st[3]); random.setstate(st[4])\n\n    def _f_keep(self, scores, clips, cfg):\n        b = cfg[\"bounds\"]\n        fs = [.5 * norm(a, b[\"countgd\"]) + .5 * norm(c, b[\"clip\"]) for a, c in zip(scores, clips)]\n        return fs, [f >= cfg[\"threshold\"] for f in fs]\n\n    # ------------------------------------------------------------ one query\n    def run_query(self, image, prompt):\n        \"\"\"Trả dict theo phương pháp. Lỗi của một phương pháp được ghi riêng, không thay bằng phương pháp khác.\"\"\"\n        torch, np, ns = self.torch, self.np, self.ns\n        out = {m: dict(ok=False, error=None, boxes=[], scores=[], count=None, timing={}, info={}) for m in METHODS}\n        torch.manual_seed(42); np.random.seed(42); ns[\"random\"].seed(42)\n        self._sync(); t = time.perf_counter()\n        b0, s0, cap0 = self._predict(image, prompt, [])\n        self._sync(); stage1_ms = (time.perf_counter() - t) * 1000\n        after = self._rng_state()\n        shared = dict(stage1_ms=round(stage1_ms, 1))\n        cache = {}\n\n        def stage2(boxes):\n            key = json.dumps(boxes)\n            if key in cache:\n                return cache[key] + (True,)\n            self._set_rng(after)\n            self._sync(); t2 = time.perf_counter()\n            final, sf, _ = self._predict(image, prompt, boxes)\n            self._sync()\n            cache[key] = (final, sf, round((time.perf_counter() - t2) * 1000, 1))\n            return cache[key] + (False,)\n\n        # B0\n        out[\"B0\"].update(ok=True, boxes=b0.tolist(), scores=s0.tolist(), count=len(b0), timing=dict(stage1_ms=shared[\"stage1_ms\"]))\n        # B2\n        try:\n            pb, ps, _, _ = select_pseudo(cap0[\"scores\"], cap0[\"boxes\"], cap0[\"ids\"], image.size, threshold=TAU, maximum=MAX_PSEUDO)\n            chosen = pb.tolist()\n            if chosen:\n                final, sf, ms, reused = stage2(chosen)\n                tim = dict(stage1_ms=shared[\"stage1_ms\"], stage2_ms=ms)\n            else:\n                final, sf = b0, s0; tim = dict(stage1_ms=shared[\"stage1_ms\"], stage2_ms=None)\n            out[\"B2\"].update(ok=True, boxes=final.tolist(), scores=sf.tolist(), count=len(final), timing=tim,\n                             info=dict(n_pseudo=len(chosen), pseudo_boxes=chosen, stage2_executed=bool(chosen)))\n        except Exception as e:  # noqa: BLE001\n            out[\"B2\"][\"error\"] = f\"{type(e).__name__}: {e}\"\n        # P(τ 0.20, M 5) rồi F\n        try:\n            t = time.perf_counter()\n            cands, _ = top_candidates(cap0[\"scores\"], cap0[\"boxes\"], cap0[\"ids\"], image.size, TAU, self.m)\n            for c, cos in zip(cands, self.clip.score(image, [c[\"box_xyxy\"] if c[\"valid\"] else [0, 0, 1, 1] for c in cands], prompt)):\n                c[\"clip_cosine\"] = cos if c[\"valid\"] else None\n            self._sync(); verify_ms = round((time.perf_counter() - t) * 1000, 1)\n            picked = choose(cands, self.tau_pseudo, MAX_PSEUDO)\n            pboxes = [c[\"box_xyxy\"] for c in picked]\n            if pboxes:\n                pf, psf, ms, reused = stage2(pboxes)\n            else:\n                pf, psf, ms, reused = b0, s0, None, False\n            t = time.perf_counter()\n            clips = self.clip.score(image, pf.tolist(), prompt)\n            self._sync(); clip_ms = round((time.perf_counter() - t) * 1000, 1)\n            fs, keep = self._f_keep(psf.tolist(), clips, self.f_main)\n            kb = [b for b, k in zip(pf.tolist(), keep) if k]; ks = [s for s, k in zip(psf.tolist(), keep) if k]\n            out[\"P+F\"].update(ok=True, boxes=kb, scores=ks, count=len(kb),\n                              timing=dict(stage1_ms=shared[\"stage1_ms\"], clip_verify_ms=verify_ms,\n                                          stage2_ms=0.0 if reused else ms, stage2_reused_from_B2=reused, clip_final_ms=clip_ms),\n                              info=dict(n_candidates=len(cands), n_pseudo=len(pboxes), stage2_executed=bool(pboxes),\n                                        same_exemplars_as_B2=reused, source_boxes=pf.tolist(), source_count=len(pf),\n                                        dropped_boxes=[b for b, k in zip(pf.tolist(), keep) if not k], f_scores=fs,\n                                        candidate_clip=[c[\"clip_cosine\"] for c in cands]))\n        except Exception as e:  # noqa: BLE001\n            out[\"P+F\"][\"error\"] = f\"{type(e).__name__}: {e}\"\n        # C6 = B0 rồi F (bounds/threshold riêng)\n        try:\n            t = time.perf_counter()\n            clips0 = self.clip.score(image, b0.tolist(), prompt)\n            self._sync(); clip_ms = round((time.perf_counter() - t) * 1000, 1)\n            fs, keep = self._f_keep(s0.tolist(), clips0, self.f_c6)\n            kb = [b for b, k in zip(b0.tolist(), keep) if k]; ks = [s for s, k in zip(s0.tolist(), keep) if k]\n            out[\"C6\"].update(ok=True, boxes=kb, scores=ks, count=len(kb), timing=dict(stage1_ms=shared[\"stage1_ms\"], clip_final_ms=clip_ms),\n                             info=dict(source_count=len(b0), dropped_boxes=[b for b, k in zip(b0.tolist(), keep) if not k], f_scores=fs))\n        except Exception as e:  # noqa: BLE001\n            out[\"C6\"][\"error\"] = f\"{type(e).__name__}: {e}\"\n        return out, shared\n\n    # ------------------------------------------------------------ request\n    def run_request(self, image, prompts, gts=None, client=None):\n        image = image.convert(\"RGB\")\n        rid = datetime.now(TZ).strftime(\"%Y%m%d-%H%M%S-\") + uuid.uuid4().hex[:6]\n        buf = __import__(\"io\").BytesIO(); image.save(buf, format=\"PNG\")\n        img_sha = hashlib.sha256(buf.getvalue()).hexdigest()\n        t = time.perf_counter()\n        per_q = []\n        for q, gt in zip(prompts, gts or [None] * len(prompts)):\n            try:\n                res, shared = self.run_query(image, q)\n                per_q.append(dict(prompt=q, gt=gt, ok=True, results=res, shared=shared, error=None))\n            except Exception as e:  # noqa: BLE001  lỗi lượt 1: cả bốn cách đều không có kết quả\n                per_q.append(dict(prompt=q, gt=gt, ok=False, results=None, shared=None, error=f\"{type(e).__name__}: {e}\"))\n        total_s = round(time.perf_counter() - t, 2)\n        # box dùng chung giữa các cặp query, theo từng cách (cùng định nghĩa với báo cáo, IoU >= 0.5)\n        xy = []\n        for i in range(len(per_q)):\n            for j in range(i + 1, len(per_q)):\n                a, b = per_q[i], per_q[j]\n                row = dict(x=a[\"prompt\"], y=b[\"prompt\"])\n                for m in METHODS:\n                    if a[\"ok\"] and b[\"ok\"] and a[\"results\"][m][\"ok\"] and b[\"results\"][m][\"ok\"]:\n                        X, Y = a[\"results\"][m][\"boxes\"], b[\"results\"][m][\"boxes\"]\n                        s = sum(shared_flags(X, Y)) + sum(shared_flags(Y, X))\n                        row[m] = (s, len(X) + len(Y))\n                    else:\n                        row[m] = None\n                xy.append(row)\n        rec = dict(request_id=rid, time=datetime.now(TZ).isoformat(timespec=\"seconds\"), inference=True,\n                   image_sha256_png=img_sha, image_size=list(image.size), prompts=prompts, gt_user=gts,\n                   env=self.env, config=self.config, total_s=total_s, client=client,\n                   queries=[dict(prompt=p[\"prompt\"], error=p[\"error\"], stage1_ms=(p[\"shared\"] or {}).get(\"stage1_ms\"),\n                                 methods={m: dict(ok=r[\"ok\"], count=r[\"count\"], error=r[\"error\"], timing=r[\"timing\"],\n                                                  n_pseudo=r[\"info\"].get(\"n_pseudo\"), same_exemplars_as_B2=r[\"info\"].get(\"same_exemplars_as_B2\"))\n                                          for m, r in (p[\"results\"] or {}).items()}) for p in per_q],\n                   xy_shared={f\"{r['x']}|{r['y']}\": {m: r[m] for m in METHODS} for r in xy})\n        with self.log_path.open(\"a\", encoding=\"utf-8\") as f:\n            f.write(json.dumps(rec, ensure_ascii=False) + \"\\n\")\n        print(f\"[inference] {rid} {image.size} {prompts} -> \" +\n              \" | \".join(f\"{p['prompt']}: \" + (\", \".join(f\"{m}={p['results'][m]['count']}\" for m in METHODS) if p[\"ok\"] else p[\"error\"]) for p in per_q)\n              + f\" ({total_s}s, {self.env['device']})\", flush=True)\n        return dict(request_id=rid, image_sha256=img_sha, image=image, queries=per_q, xy=xy, total_s=total_s, log=rec)\n", "app_live.py": "\"\"\"Demo suy luận THẬT — CS331: dán ảnh mới, nhập tên vật, chạy CountGD++ (B0, B2, P+F, C6) với cấu hình đã khóa.\n\nChạy trên GPU (Kaggle T4, xem kaggle_live_demo.ipynb):\n    python demo/live/app_live.py --device cuda --share --password <mật khẩu>\nKiểm chức năng trên CPU (chậm, không dùng để trình bày):\n    .venv-live\\\\Scripts\\\\python demo\\\\live\\\\app_live.py --device cpu --port 7870\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nos.environ.setdefault(\"GRADIO_ANALYTICS_ENABLED\", \"False\")\nfor _s in (sys.stdout, sys.stderr):\n    try:\n        _s.reconfigure(encoding=\"utf-8\", errors=\"replace\")\n    except (AttributeError, ValueError):\n        pass\nsys.path.insert(0, str(Path(__file__).resolve().parent))\n\nimport live_engine as le  # noqa: E402\nfrom PIL import Image, ImageDraw, ImageFont  # noqa: E402\n\nMAX_Q = 3\nCAP = 900  # số box đề xuất tối đa mỗi ảnh của CountGD++ (num_queries trong cfg_app.py)\nMAX_W = 760\nCOLORS = {\"B0\": (31, 119, 180), \"B2\": (214, 39, 40), \"P+F\": (44, 160, 44), \"C6\": (148, 103, 189)}\nDESC = {\"B0\": \"CountGD++ một lượt\", \"B2\": \"CountGD++ hai lượt (pseudo-exemplar gốc)\",\n        \"P+F\": \"Cải tiến đã khóa: P(τ 0.20, M 5) rồi lọc F\", \"C6\": \"Một lượt rồi lọc F (ngưỡng khóa riêng)\"}\nENGINE: le.LiveEngine | None = None\n\n# Ctrl+V ở bất kỳ đâu trên trang (trừ khi đang gõ chữ) -> đưa ảnh trong clipboard vào ô ảnh.\nPASTE_JS = r\"\"\"\n<script>\n(function () {\n  async function putFile(file) {\n    const root = document.querySelector('#live_image');\n    if (!root) return;\n    let input = root.querySelector('input[type=file]');\n    if (!input) {                       // ô đang có ảnh cũ: bấm nút xóa của Gradio rồi chờ ô upload hiện lại\n      const clear = [...root.querySelectorAll('button')].find(b => /clear|xóa|remove/i.test((b.getAttribute('aria-label') || '') + (b.title || '')));\n      if (clear) clear.click();\n      for (let i = 0; i < 30 && !input; i++) { await new Promise(r => setTimeout(r, 100)); input = root.querySelector('input[type=file]'); }\n    }\n    if (!input) { alert('Không tìm thấy ô ảnh để dán. Bấm \"Xóa và bắt đầu lại\" rồi thử Ctrl+V lần nữa.'); return; }\n    const dt = new DataTransfer();\n    dt.items.add(new File([file], file.name && file.name !== 'image.png' ? file.name : 'pasted.png', { type: file.type || 'image/png' }));\n    input.files = dt.files;\n    input.dispatchEvent(new Event('change', { bubbles: true }));\n  }\n  window.addEventListener('paste', function (e) {\n    const t = e.target;\n    const typing = t && ((t.tagName === 'TEXTAREA') || (t.tagName === 'INPUT' && t.type !== 'file') || t.isContentEditable);\n    const files = [...((e.clipboardData && e.clipboardData.files) || [])].filter(f => f.type.startsWith('image/'));\n    if (!files.length || typing) return;      // dán chữ vào ô chữ vẫn bình thường\n    e.preventDefault(); e.stopPropagation();\n    putFile(files[0]);\n  }, true);\n})();\n</script>\n\"\"\"\n\n\ndef font(size):\n    for f in (\"arialbd.ttf\", \"arial.ttf\", \"DejaVuSans-Bold.ttf\", \"/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf\"):\n        try:\n            return ImageFont.truetype(f, size)\n        except OSError:\n            pass\n    return ImageFont.load_default()\n\n\ndef render(image, kept, dropped, color, caption):\n    s = MAX_W / image.width\n    im = image.resize((MAX_W, max(1, round(image.height * s))), Image.LANCZOS) if s != 1 else image.copy()\n    d = ImageDraw.Draw(im)\n    for b in dropped or []:\n        x0, y0, x1, y1 = (v * s for v in b)\n        for xa in range(int(x0), int(x1), 8):\n            d.line([(xa, y0), (min(xa + 4, x1), y0)], fill=(160, 160, 160), width=2)\n            d.line([(xa, y1), (min(xa + 4, x1), y1)], fill=(160, 160, 160), width=2)\n        for ya in range(int(y0), int(y1), 8):\n            d.line([(x0, ya), (x0, min(ya + 4, y1))], fill=(160, 160, 160), width=2)\n            d.line([(x1, ya), (x1, min(ya + 4, y1))], fill=(160, 160, 160), width=2)\n    for b in kept:\n        d.rectangle([v * s for v in b], outline=color, width=3)\n    f = font(max(16, im.width // 30)); h = f.size + 12\n    d.rectangle([0, im.height - h, im.width, im.height], fill=(255, 255, 255))\n    d.text((8, im.height - h + 5), caption, fill=(20, 20, 20), font=f)\n    return im\n\n\ndef fmt_ms(v):\n    return \"—\" if v is None else (f\"{v / 1000:.2f} s\" if v >= 1000 else f\"{v:.0f} ms\")\n\n\ndef timing_text(m, r):\n    t = r[\"timing\"]\n    if m == \"B0\":\n        return f\"lượt 1 {fmt_ms(t.get('stage1_ms'))} (dùng chung)\"\n    if m == \"B2\":\n        return (f\"lượt 1 (chung) + lượt 2 {fmt_ms(t.get('stage2_ms'))}\" if t.get(\"stage2_ms\") is not None\n                else \"lượt 1 (chung); không có pseudo-exemplar → bằng B0\")\n    if m == \"P+F\":\n        s2 = (\"lượt 2 dùng lại của B2 (cùng bộ exemplar)\" if t.get(\"stage2_reused_from_B2\")\n              else (f\"lượt 2 {fmt_ms(t.get('stage2_ms'))}\" if t.get(\"stage2_ms\") is not None else \"không còn exemplar → nguồn = B0\"))\n        return f\"lượt 1 (chung) + CLIP kiểm exemplar {fmt_ms(t.get('clip_verify_ms'))} + {s2} + CLIP chấm box {fmt_ms(t.get('clip_final_ms'))}\"\n    return f\"lượt 1 (chung) + CLIP chấm box {fmt_ms(t.get('clip_final_ms'))}\"\n\n\n# ---------------------------------------------------------------- callbacks\ndef n_outputs():\n    return MAX_Q * (1 + 1 + 4 + 1) + 3   # mỗi query: group, tiêu đề, 4 ảnh, bảng; + status, xy, log\n\n\ndef blank(status):\n    import gradio as gr\n    outs = []\n    for _ in range(MAX_Q):\n        outs += [gr.Group(visible=False), gr.Markdown(value=\"\")] + [gr.Image(value=None, label=m) for m in le.METHODS] + [gr.Markdown(value=\"\")]\n    return outs + [gr.Markdown(value=status), gr.Markdown(value=\"\"), gr.JSON(value=None)]\n\n\ndef on_input_change(image, queries, gts):\n    msg = (\"⚠ **Ảnh hoặc tên vật đã thay đổi — kết quả cũ đã được xóa.** Bấm **Chạy** để chạy model trên dữ liệu mới.\"\n           if image is not None else \"Dán ảnh (Ctrl+V), kéo thả hoặc bấm vào ô để chọn ảnh, nhập tên vật rồi bấm **Chạy**.\")\n    return blank(msg)\n\n\ndef start_run(image, queries, gts):\n    return blank(\"⏳ **Đang chạy model…** Kết quả cũ đã được xóa. Trên GPU T4 mỗi tên vật mất vài giây.\")\n\n\ndef run(image, queries, gts, request=None):\n    import gradio as gr\n    if image is None:\n        return blank(\"❌ Chưa có ảnh. Dán ảnh bằng **Ctrl+V**, kéo thả, hoặc bấm vào ô ảnh để chọn file.\")\n    try:\n        prompts = le.parse_queries(queries, MAX_Q)\n        gt = le.parse_gt(gts, len(prompts))\n    except ValueError as e:\n        return blank(f\"❌ {e}\")\n    try:\n        res = ENGINE.run_request(image, prompts, gt, client=getattr(getattr(request, \"client\", None), \"host\", None))\n    except Exception as e:  # noqa: BLE001\n        return blank(f\"❌ Lỗi khi chạy model: `{type(e).__name__}: {e}`. Kết quả không được hiển thị.\")\n    outs = []\n    for qi in range(MAX_Q):\n        if qi >= len(res[\"queries\"]):\n            outs += [gr.Group(visible=False), gr.Markdown(value=\"\")] + [gr.Image(value=None) for _ in le.METHODS] + [gr.Markdown(value=\"\")]\n            continue\n        q = res[\"queries\"][qi]\n        imgs, rows, capped = [], [], False\n        for m in le.METHODS:\n            if not q[\"ok\"]:\n                imgs.append(gr.Image(value=None, label=f\"{m} — lỗi lượt 1\")); continue\n            r = q[\"results\"][m]\n            if not r[\"ok\"]:\n                imgs.append(gr.Image(value=None, label=f\"{m} — LỖI: {r['error']}\"))\n                rows.append(f\"| **{m}** | lỗi | — | `{r['error']}` |\")\n                continue\n            dropped = r[\"info\"].get(\"dropped_boxes\")\n            cap = f\"{m} · \\\"{q['prompt']}\\\": {r['count']}\" + (\"\" if q[\"gt\"] is None else f\" (GT {q['gt']})\")\n            imgs.append(gr.Image(value=render(res[\"image\"], r[\"boxes\"], dropped, COLORS[m], cap),\n                                 label=f\"{m} — {DESC[m]}\"))\n            err = \"—\" if q[\"gt\"] is None else str(abs(r[\"count\"] - q[\"gt\"]))\n            extra = \"\"\n            if m in (\"P+F\", \"C6\"):\n                extra = f\" · F giữ {r['count']}/{r['info']['source_count']} box\"\n            if m in (\"B2\", \"P+F\") and r[\"info\"].get(\"n_pseudo\") is not None:\n                extra += f\" · {r['info']['n_pseudo']} pseudo-exemplar\"\n            if max(r[\"count\"], r[\"info\"].get(\"source_count\") or 0) >= CAP:\n                extra += f\" · ⚠ chạm trần {CAP} box\"\n                capped = True\n            rows.append(f\"| **{m}** | **{r['count']}**{extra} | {err} | {timing_text(m, r)} |\")\n        head = f\"### Tên vật {qi + 1}: `{q['prompt']}`\" + (\"\" if q[\"gt\"] is None else f\" · GT bạn nhập: {q['gt']}\")\n        if capped:\n            head += (f\"\\n\\n⚠ **Có cách chạm trần {CAP} box**: CountGD++ chỉ có {CAP} box đề xuất mỗi ảnh, nên con số này là giới hạn \"\n                     \"của mô hình, không phải số vật thật. Nên dùng ảnh ít vật hơn (dữ liệu của đồ án: 3–40 vật mỗi class).\")\n        if not q[\"ok\"]:\n            table = f\"❌ Lỗi ở lượt 1, cả bốn cách đều không có kết quả: `{q['error']}`\"\n        else:\n            table = (f\"Lượt 1 (CountGD++ chỉ với tên vật) chạy **một lần** và dùng chung cho cả bốn cách: \"\n                     f\"{fmt_ms(q['shared']['stage1_ms'])}.\\n\\n| Cách | Số đếm | Sai số so với GT bạn nhập | Thời gian |\\n|---|---|---|---|\\n\"\n                     + \"\\n\".join(rows))\n        outs += [gr.Group(visible=True), gr.Markdown(value=head)] + imgs + [gr.Markdown(value=table)]\n    xy_md = \"\"\n    if res[\"xy\"]:\n        lines = [\"### Đếm lẫn giữa các tên vật (box dùng chung, IoU ≥ 0.5)\",\n                 \"Tỉ lệ box của hai query có box trùng ở query kia. Cao nghĩa là đổi tên vật mà mô hình vẫn khoanh cùng những vật đó.\",\n                 \"\", \"| Cặp | \" + \" | \".join(le.METHODS) + \" |\", \"|---|\" + \"---|\" * len(le.METHODS)]\n        for r in res[\"xy\"]:\n            cells = [(\"lỗi\" if r[m] is None else (f\"{r[m][0]}/{r[m][1]} = {r[m][0] / r[m][1]:.0%}\" if r[m][1] else \"0 box\")) for m in le.METHODS]\n            lines.append(f\"| `{r['x']}` / `{r['y']}` | \" + \" | \".join(cells) + \" |\")\n        xy_md = \"\\n\".join(lines)\n    env = ENGINE.env\n    status = (f\"✅ **Đã chạy model thật** trên ảnh vừa nhập ({res['image'].width}×{res['image'].height}) · \"\n              f\"request `{res['request_id']}` · {res['total_s']} s · {env['device']}. \"\n              + (\"Không có GT nên không tính sai số.\" if all(q[\"gt\"] is None for q in res[\"queries\"]) else\n                 \"Sai số chỉ tính cho ví dụ này, theo GT bạn nhập; không phải kết quả nghiên cứu.\"))\n    return outs + [gr.Markdown(value=status), gr.Markdown(value=xy_md), gr.JSON(value=res[\"log\"])]\n\n\ndef signature(image, queries, gts):\n    \"\"\"Dấu vân tay của đầu vào: dùng để phát hiện ảnh/tên vật bị đổi trong lúc model đang chạy.\"\"\"\n    import hashlib\n    if image is None:\n        return None\n    return hashlib.md5(image.convert(\"RGB\").resize((64, 64)).tobytes()).hexdigest() + \"|\" + (queries or \"\") + \"|\" + (gts or \"\")\n\n\ndef stale_check(image, queries, gts, used):\n    import gradio as gr\n    if used is not None and signature(image, queries, gts) != used:\n        return blank(\"⚠ **Ảnh hoặc tên vật đã đổi trong lúc model đang chạy — đã bỏ kết quả của dữ liệu cũ.** Bấm **Chạy** lại.\")\n    return [gr.skip()] * n_outputs()\n\n\ndef build():\n    import gradio as gr\n    e = ENGINE\n    cpu_warn = (\"\\n\\n⚠ **Đang chạy trên CPU — chỉ để kiểm chức năng, rất chậm (vài chục giây mỗi tên vật).** \"\n                \"Demo trình bày chạy trên GPU Kaggle T4.\" if e.device.type == \"cpu\" else \"\")\n    with gr.Blocks(title=\"CS331 · Demo đếm vật theo tên (CountGD++)\", theme=gr.themes.Soft(), head=PASTE_JS,\n                   analytics_enabled=False) as demo:\n        gr.Markdown(\"# Đếm vật theo tên trên ảnh mới — CountGD++ và cải tiến của nhóm (CS331)\")\n        gr.Markdown(f\"Model chạy **thật** trên ảnh bạn đưa vào: `{e.env['device']}` · checkpoint `{e.ckpt_sha[:12]}…` (đã audit) · \"\n                    f\"freeze v2 `{le.FREEZE_SHA[:8]}…` · P(τ {e.tau_pseudo:.2f}, M {e.m}) · F: P+F θ = {e.f_main['threshold']:.3f}, \"\n                    f\"C6 θ = {e.f_c6['threshold']:.3f} (bounds riêng). Ảnh mới không có GT, nên app **không** tính MAE hay khẳng định đếm đúng.\"\n                    + cpu_warn)\n        with gr.Row():\n            with gr.Column(scale=3):\n                img = gr.Image(label=\"Ảnh — Ctrl+V để dán, kéo thả, hoặc bấm để chọn file\", type=\"pil\",\n                               sources=[\"upload\", \"clipboard\"], elem_id=\"live_image\", height=360)\n            with gr.Column(scale=2):\n                qbox = gr.Textbox(label=\"Tên vật cần đếm (tiếng Anh, số nhiều; tối đa 3, cách nhau bởi dấu phẩy)\",\n                                  placeholder=\"bowls, plates\", lines=1, elem_id=\"live_queries\")\n                gr.Markdown(\"Ví dụ: `bowls, plates` · `people, cars` · `apples` · `dogs, sheep`\")\n                gtbox = gr.Textbox(label=\"GT (tùy chọn) — số vật thật theo cùng thứ tự, '-' nếu không biết\",\n                                   placeholder=\"3, 16\", lines=1, elem_id=\"live_gt\")\n                with gr.Row():\n                    run_btn = gr.Button(\"Chạy\", variant=\"primary\", elem_id=\"live_run\")\n                    clear_btn = gr.Button(\"Xóa và bắt đầu lại\", elem_id=\"live_clear\")\n        status = gr.Markdown(\"Dán ảnh (Ctrl+V), kéo thả hoặc bấm vào ô để chọn ảnh, nhập tên vật rồi bấm **Chạy**.\", elem_id=\"live_status\")\n        groups = []\n        for qi in range(MAX_Q):\n            with gr.Group(visible=False) as g:\n                hd = gr.Markdown()\n                with gr.Row():\n                    ims = [gr.Image(label=m, type=\"pil\", interactive=False, show_download_button=True) for m in le.METHODS[:2]]\n                with gr.Row():\n                    ims += [gr.Image(label=m, type=\"pil\", interactive=False, show_download_button=True) for m in le.METHODS[2:]]\n                tbl = gr.Markdown()\n            groups += [g, hd] + ims + [tbl]\n        xy = gr.Markdown()\n        with gr.Accordion(\"Log của lần chạy này (xác nhận đã chạy inference + cấu hình dùng)\", open=False):\n            log = gr.JSON()\n        outs = groups + [status, xy, log]\n        ins = [img, qbox, gtbox]\n        # ảnh: change (upload/dán/xóa); ô chữ: input (chỉ khi người dùng gõ), để không chen vào sau nút Chạy\n        img.change(on_input_change, ins, outs, queue=False)\n        qbox.input(on_input_change, ins, outs, queue=False)\n        gtbox.input(on_input_change, ins, outs, queue=False)\n        used = gr.State(None)\n        remember = lambda i, q, g: signature(i, q, g)  # noqa: E731\n        for trig in (run_btn.click, qbox.submit):\n            (trig(start_run, ins, outs, queue=False)\n             .then(remember, ins, used, queue=False)\n             .then(run, ins, outs)\n             .then(stale_check, ins + [used], outs, queue=False))\n\n        def clear_all():\n            return [gr.Image(value=None), gr.Textbox(value=\"\"), gr.Textbox(value=\"\")] + blank(\n                \"Đã xóa. Dán ảnh mới (Ctrl+V), nhập tên vật rồi bấm **Chạy**.\")\n        clear_btn.click(clear_all, None, ins + outs, queue=False)\n    return demo\n\n\ndef main():\n    global ENGINE\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--device\", default=\"cuda\", choices=[\"cuda\", \"cpu\"])\n    ap.add_argument(\"--host\", default=\"127.0.0.1\")\n    ap.add_argument(\"--port\", type=int, default=7860)\n    ap.add_argument(\"--share\", action=\"store_true\", help=\"Tạo link https *.gradio.live (dùng khi model chạy trên Kaggle)\")\n    ap.add_argument(\"--password\", default=os.environ.get(\"DEMO_PASSWORD\"), help=\"Bật đăng nhập (user: cs331)\")\n    ap.add_argument(\"--log-dir\", default=None)\n    a = ap.parse_args()\n    t = time.perf_counter()\n    ENGINE = le.LiveEngine(a.device, a.log_dir)\n    print(f\"Model + CLIP đã nạp ({ENGINE.load_s}s) trên {ENGINE.env['device']}; log: {ENGINE.log_path}\", flush=True)\n    demo = build()\n    demo.queue(default_concurrency_limit=1).launch(\n        server_name=a.host, server_port=a.port, share=a.share, inbrowser=False, show_api=False,\n        auth=(\"cs331\", a.password) if a.password else None, max_file_size=\"20mb\")\n    print(f\"Dừng sau {time.perf_counter() - t:.0f}s\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n", "check_against_test.py": "\"\"\"Kiểm wrapper suy luận: chạy LiveEngine trên vài query test đã có kết quả chính thức và so số đếm 4 cách.\n\nKHÔNG phải chạy lại test, không dùng để báo cáo: chỉ xác nhận wrapper ghép đúng pipeline đã khóa\n(lượt 1, B2, P τ 0.20 M 5, F theo freeze). Trên Kaggle T4 (cùng môi trường lần test) số đếm phải trùng;\ntrên CPU có thể lệch nhẹ do số học khác GPU, nên CPU chỉ báo lệch, không coi là lỗi.\n\nDùng: python demo/live/check_against_test.py --device cuda --data-root <thư mục có data_external> [--queries t01_1 t01_2 t05_1]\n\"\"\"\nimport argparse\nimport csv\nimport json\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport live_engine as le  # noqa: E402\nfrom PIL import Image  # noqa: E402\n\nRESULT = le.ROOT / \"results/countgdpp/p1_test_kaggle_t4_20261111_1430_eval/result.json\"\nMANIFEST = le.ROOT / \"experiments/countgdpp/dev_test_v2/test/manifest.csv\"\nKEYS = {\"B0\": \"B0\", \"B2\": \"B2\", \"P+F\": \"P(tau 0.20, M 5)+F\", \"C6\": \"C6 = B0+F\"}\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--device\", default=\"cuda\", choices=[\"cuda\", \"cpu\"])\n    ap.add_argument(\"--data-root\", default=str(le.ROOT))\n    ap.add_argument(\"--queries\", nargs=\"+\", default=[\"t01_1\", \"t01_2\", \"t05_1\"])\n    ap.add_argument(\"--log-dir\", default=None)\n    a = ap.parse_args()\n    res = json.loads(RESULT.read_text(encoding=\"utf-8\"))[\"per_query_counts\"]\n    man = {r[\"query_id\"]: r for r in csv.DictReader(open(MANIFEST, encoding=\"utf-8\", newline=\"\"))}\n    eng = le.LiveEngine(a.device, Path(a.log_dir) if a.log_dir else le.ROOT / \"demo/live/logs_check\")\n    print(f\"Engine: {eng.env['device']} · nạp {eng.load_s}s\", flush=True)\n    ok_all = True\n    for q in a.queries:\n        r = man[q]\n        image = Image.open(Path(a.data_root) / r[\"image_path\"]).convert(\"RGB\")\n        out, shared = eng.run_query(image, r[\"prompt\"])\n        got = {m: out[m][\"count\"] for m in le.METHODS}\n        want = {m: res[KEYS[m]][q] for m in le.METHODS}\n        same = got == want\n        ok_all &= same\n        print(f\"{q} '{r['prompt']}': mới {got} | chính thức {want} | {'KHỚP' if same else 'LỆCH'} · lượt 1 {shared['stage1_ms']:.0f} ms\"\n              + \"\".join(f\" · {m}: {out[m]['error']}\" for m in le.METHODS if out[m][\"error\"]), flush=True)\n    print(\"CHECK:\", \"PASS (khớp hết)\" if ok_all else (\"LỆCH — trên GPU T4 là lỗi, phải xem lại\" if a.device == \"cuda\"\n                                                       else \"LỆCH trên CPU — có thể do số học CPU khác GPU; kiểm lại trên Kaggle T4\"))\n    raise SystemExit(0 if ok_all or a.device == \"cpu\" else 1)\n\n\nif __name__ == \"__main__\":\n    main()\n"}
DEMO_SHA = {"live_engine.py": "05d4d3fc1495f4f7d6e2bc71d73e9b3b011b398a6b0a8e43ffcaefabf7ddef97", "app_live.py": "2f11e3af6404da90e046a9b33191ba2485f8221c9d7fa1e584b46181ae22f8a3", "check_against_test.py": "7d665b3e155b02c1fc568733e80dfe5097f3ee7980c56534a9a5e367f9be46a7"}
live = ROOT / "demo/live"; live.mkdir(parents=True, exist_ok=True)
for name, src in DEMO_FILES.items():
    (live / name).write_text(src, encoding="utf-8")
    assert hashlib.sha256(src.encode("utf-8")).hexdigest() == DEMO_SHA[name], name
    print(f"{name}: sha256 {DEMO_SHA[name][:12]}…")

In [ ]:
# Ô 6 — kiểm wrapper trên GPU: 3 query test đã có kết quả chính thức phải ra đúng số đếm của cả 4 cách.
# Đây KHÔNG phải chạy lại test (không chấm, không ghi vào kết quả); dừng nếu lệch.
sh(f"{PY} demo/live/check_against_test.py --device cuda --data-root {ROOT} --queries t01_1 t01_2 t05_1 --log-dir {LOGS}/check", cwd=ROOT)

In [ ]:
# Ô 7 — MỞ DEMO. Giữ ô này chạy trong lúc demo; bấm Stop (■) để tắt.
proc = subprocess.Popen([PY, "demo/live/app_live.py", "--device", "cuda", "--share", "--host", "0.0.0.0", "--port", "7860",
                         "--password", DEMO_PASSWORD, "--log-dir", str(LOGS)], cwd=ROOT, env=ENV,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = None
try:
    for line in proc.stdout:
        if "Warning" in line or "warn(" in line:
            continue
        print(line, end="", flush=True)
        m = re.search(r"https://[\w.-]+\.gradio\.live", line)
        if m and not url:
            url = m.group(0)
            print("\n" + "=" * 70 + f"\nLINK DEMO: {url}\nĐăng nhập: user = cs331 · mật khẩu = {DEMO_PASSWORD}\n" + "=" * 70, flush=True)
except KeyboardInterrupt:
    pass
finally:
    proc.terminate()
    print("Đã tắt demo. Log:", LOGS / "requests.jsonl")

## Sau khi demo

- Log mọi lần bấm Chạy: `/kaggle/working/demo_logs/requests.jsonl`. Tải ở tab Output nếu muốn lưu.
- Mỗi dòng log ghi: request id, sha256 ảnh, tên vật, thiết bị GPU, checkpoint, freeze, cấu hình, số đếm và thời gian từng cách.
- Ảnh demo **không** phải dữ liệu nghiên cứu; không đưa vào bảng kết quả.